# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the required dependencies for this notebook.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [1]:
# !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [7]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader, WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [4]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)


## Point at the new dataset

`rapeoflucrece.txt` (the full text of Shakespeare's poem, public domain) sits in the same folder
as this notebook, so `TextLoader` can reference it directly by filename — no need to search the
filesystem for it the way the demo did for `state_of_the_union.txt`.

In [5]:
doc_path = "rapeoflucrece.txt"

with open(doc_path, "r", encoding="utf-8") as f:
    print(f.read()[:400])  # sanity check: confirms the file loaded and looks right

	THE RAPE OF LUCRECE

TO THE
RIGHT HONORABLE HENRY WRIOTHESLY,
Earl of Southampton, and Baron of Tichfield.


The love I dedicate to your lordship is without end; whereof
this pamphlet, without beginning, is but a superfluous moiety.
The warrant I have of your honourable disposition, not the worth
of my untutored lines, makes it assured of acceptance. What I
have done is yours; what I have to do i


## Ingest the poem into a vector store

Three steps, same as the demo:

1. **Load** — `TextLoader` reads the raw file into a single LangChain `Document`.
2. **Split** — `CharacterTextSplitter` breaks that one long document into ~1000-character chunks
   (with no overlap) so each chunk is small enough to embed meaningfully and retrieve precisely —
   embedding the entire 2,000+ line poem as one vector would blur every stanza together.
3. **Embed + store** — `OpenAIEmbeddings` turns each chunk into a vector, and `Chroma.from_documents`
   stores them under a dedicated `collection_name` (`"rape-of-lucrece"`) so this collection doesn't
   collide with any other vector store in the same Chroma instance.

In [10]:
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

lucrece_db = Chroma.from_documents(texts, embeddings, collection_name="rape-of-lucrece")

print(f"Split into {len(texts)} chunks")

Created a chunk of size 2333, which is longer than the specified 1000


Split into 91 chunks


## Wrap the vector store in a `RetrievalQA` chain

A `RetrievalQA` chain is what actually answers a question: given a query, it retrieves the most
similar chunks from `lucrece_db`, stuffs them into the LLM's context (`chain_type="stuff"`), and
asks the LLM to answer using only that retrieved context. This chain — not the raw vector store —
is what gets wrapped as a `Tool` for the agent

In [11]:
lucrece_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=lucrece_db.as_retriever()
)

## Add a second knowledge source (Ruff docs)

To actually test tool *routing*, the agent needs more than one tool to choose between. This reuses
the demo's second source — the Ruff linter FAQ page — ingested the same way, but via `WebBaseLoader`
instead of `TextLoader` since it's a live web page rather than a local file.

In [12]:
ruff_loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")
ruff_docs = ruff_loader.load()
ruff_texts = text_splitter.split_documents(ruff_docs)

ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


## Wrap both chains as `Tool`s and build the agent

Each `Tool` needs a `name` and, more importantly, a `description` — this description is the *only*
signal the agent uses to decide which tool fits a given question, so it has to be specific about
the subject matter each tool covers. `AgentType.ZERO_SHOT_REACT_DESCRIPTION` is the same ReAct-style
agent used throughout the demo: at each step it reasons in Thought → Action → Action Input →
Observation, picking whichever tool's description best matches the current question.

In [13]:
from langchain.agents import AgentType, Tool, initialize_agent

tools = [
    Tool(
        name="Lucrece QA System",
        func=lucrece_qa.run,
        description=(
            "useful for when you need to answer questions about Shakespeare's narrative poem "
            "'The Rape of Lucrece' — its plot, characters (e.g. Lucrece, Tarquin, Collatine), "
            "themes, or specific lines. Input should be a fully formed question."
        ),
    ),
    Tool(
        name="Ruff QA System",
        func=ruff_qa.run,
        description=(
            "useful for when you need to answer questions about ruff (a python linter). "
            "Input should be a fully formed question."
        ),
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18036\1448050941.py:23: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


## Test — a question only the Lucrece tool can answer

Watch the `verbose=True` trace: the `Thought` should reference the poem/Shakespeare, the `Action`
should be `Lucrece QA System`, and the `Observation` should come back with plot detail pulled from
the retrieved chunks.

In [14]:
agent.invoke(
    "In 'The Rape of Lucrece', what does Lucrece do after Tarquin assaults her?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Lucrece QA System to answer this question.
Action: Lucrece QA System
Action Input: In 'The Rape of Lucrece', what does Lucrece do after Tarquin assaults her?
Observation:  After Tarquin assaults her, Lucrece speaks to her maid and asks her when Tarquin left. She also expresses her fear and helplessness in the situation.
Thought: I now know the final answer.
Final Answer: After Tarquin assaults her, Lucrece speaks to her maid and expresses her fear and helplessness in the situation.

> Finished chain.


{'input': "In 'The Rape of Lucrece', what does Lucrece do after Tarquin assaults her?",
 'output': 'After Tarquin assaults her, Lucrece speaks to her maid and expresses her fear and helplessness in the situation.'}

## Test — a question only the Ruff tool can answer

Same question as the original demo, unchanged — it's still a good control to confirm the agent
doesn't default to Lucrece just because it's listed first.

In [15]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 It's important to understand the differences between tools before deciding which one to use.
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"
Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.
Thought: It seems like Flake8 may be more flexible and customizable, but Ruff may have better performance and more built-in features.
Action: Ruff QA System
Action Input: "What are the advantages of using ruff over flake8?"
Observation:  Ruff has a larger rule set, supports automatic fixing of lint violations, and does not require the installation of additional plugins. It also has better compatibility with Black and can be used as a formatter as well as a lin

{'input': 'Why use ruff over flake8?',
 'output': 'Based on these observations, it seems like Ruff may be a better choice for projects that prioritize performance and convenience, while Flake8 may be better for projects that require more flexibility and customization. Ultimately, the best tool to use will depend on the specific needs and preferences of the project and its developers.'}

## Use the agent solely as a router (`return_direct=True`)

Setting `return_direct=True` on a `Tool` tells the agent: once this tool has been called, stop
reasoning and hand its output straight back as the final answer — skip the extra "I now know the
final answer" synthesis step. Useful when the retrieved answer is already complete and you don't
want the LLM paraphrasing (or drifting from) it.

In [16]:
tools_direct = [
    Tool(
        name="Lucrece QA System",
        func=lucrece_qa.run,
        description=(
            "useful for when you need to answer questions about Shakespeare's narrative poem "
            "'The Rape of Lucrece'. Input should be a fully formed question."
        ),
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff_qa.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

agent_router = initialize_agent(
    tools_direct, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

agent_router.invoke("In 'The Rape of Lucrece', who is Collatine?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Lucrece QA System to answer this question.
Action: Lucrece QA System
Action Input: Who is Collatine?
Observation:  Collatine is the husband of Lucrece, who is speaking in this passage.


> Finished chain.


{'input': "In 'The Rape of Lucrece', who is Collatine?",
 'output': ' Collatine is the husband of Lucrece, who is speaking in this passage.'}

## Multi-hop question — combine both tools

The two knowledge domains (a 16th-century poem and a Python linter) don't actually overlap in
content, so a genuine multi-hop *fact* isn't possible — but the agent can still be asked a
compound question that forces it to call **both** tools in sequence and merge their answers,
exactly like the Jupyter/Ruff example in the original demo. This uses the non-`return_direct`
agent from step 7, since combining results requires the extra reasoning step.

In [17]:
agent.invoke(
    "Who is Tarquin in 'The Rape of Lucrece', and separately, why might a Python project "
    "choose ruff over flake8?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should think about what information I need to answer these questions.
Action: Lucrece QA System
Action Input: Who is Tarquin in 'The Rape of Lucrece'?
Observation:  Tarquin is a lustful Roman who leaves the Roman army to pursue Lucrece, the chaste wife of Collatine.
Thought: I should now think about why a Python project might choose ruff over flake8.
Action: Ruff QA System
Action Input: Why might a Python project choose ruff over flake8?
Observation: 
A Python project might choose Ruff over Flake8 for a few reasons:

1. Ruff has a larger rule set: Ruff implements over 900 rules, while Flake8 only implements around 80. This means that Ruff can catch more potential issues in your code.

2. Ruff is faster: Ruff is written in Rust, which is a faster language than Python. This means that Ruff can lint your code more quickly than Flake8.

3. Ruff has a built-in formatter: Ruff comes with its own formatter, which can automatically fix some of the lint violations it finds. This can save tim

{'input': "Who is Tarquin in 'The Rape of Lucrece', and separately, why might a Python project choose ruff over flake8?",
 'output': 'Tarquin is a lustful Roman who leaves the Roman army to pursue Lucrece, the chaste wife of Collatine. A Python project might choose Ruff over Flake8 for its larger rule set, faster speed, built-in formatter, ease of installation, and support for more Python versions.'}

##  Reflection

- **Tool selection** — the agent correctly picked `Lucrece QA System` for the plot/character
  questions and `Ruff QA System` for the linter question, because each tool's `description` names
  its subject matter explicitly; there's no ambiguity for the router to resolve.
- **`return_direct=True` vs. default** — with `return_direct=True`, the chain's raw answer *is*
  the final answer, so the trace is shorter (one Action/Observation, no extra synthesis). Without
  it, the agent can keep reasoning afterward, which is exactly what's needed for the multi-hop
  question in step 11 — `return_direct` would have made the agent stop after the *first* tool call
  and never reach the second.
- **Takeaway** — the pattern from the demo transfers directly to a new dataset: swap the loader,
  give the new `Tool` an accurate `description`, and the ReAct agent handles routing without any
  other changes.